# Solar Panel Damage Detection — YOLOv5 Training

This notebook trains a YOLOv5 model to detect three types of solar panel damage:
- **Class 0** — Hotspots / Burned Cells (small defects, benefits from high resolution)
- **Class 1** — Broken / Cracks
- **Class 2** — Soiling


**Precision target:** ≥ 75 % precision is required for the model to be acceptable
for solar panel damage recognition.

**Key optimizations applied:**
- `--img 1024` — higher resolution improves detection of small Class 0 defects
- `--weights yolov5l.pt` — large model for better feature extraction, targets ≥ 75 % precision
- `--optimizer AdamW` — stable convergence on small datasets
- `--freeze 10` — freezes first 10 backbone layers to preserve pre-trained features
- Custom hyperparameters (`hyp.scratch_custom.yaml`) with heavy augmentation and label smoothing
- `--cache ram` — caches all images in RAM to eliminate disk I/O bottlenecks and speed up each epoch
- `--workers 8` — uses 8 parallel data-loader workers to maximize GPU utilization

## Step 1 — Check GPU

In [ ]:
!nvidia-smi

## Step 2 — Clone YOLOv5 and install requirements

In [ ]:
import os

# Clone official YOLOv5 repository
if not os.path.exists('/content/yolov5'):
    !git clone https://github.com/ultralytics/yolov5 /content/yolov5
else:
    print('yolov5 already cloned')

%cd /content/yolov5
!pip install -r requirements.txt -q

## Step 3 — Clone the dataset repository

In [ ]:
if not os.path.exists('/content/Activity_yolov5'):
    !git clone https://github.com/Carfok/Activity_yolov5 /content/Activity_yolov5
else:
    print('Activity_yolov5 already cloned')

## Step 4 — Verify dataset and class distribution

In [ ]:
%cd /content/Activity_yolov5
!python create_dataset.py

## Step 5 — Confirm dataset config paths

In [ ]:
import yaml

with open('/content/Activity_yolov5/personalizado.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
print('Dataset config:')
for k, v in cfg.items():
    print(f'  {k}: {v}')

# Verify that the image directories exist
from pathlib import Path
train_dir = Path(cfg.get('path', '')) / cfg.get('train', '')
val_dir   = Path(cfg.get('path', '')) / cfg.get('val', '')
print(f'\nTrain dir exists: {train_dir.exists()} → {train_dir}')
print(f'Val   dir exists: {val_dir.exists()} → {val_dir}')

## Step 6 — Copy custom hyperparameters into YOLOv5 data directory

In [ ]:
import shutil

src = '/content/Activity_yolov5/hyp.scratch_custom.yaml'
dst = '/content/yolov5/data/hyps/hyp.scratch_custom.yaml'
shutil.copy(src, dst)
print(f'Copied {src} → {dst}')

## Step 7 — Train YOLOv5

Key flags:
| Flag | Value | Reason |
|------|-------|--------|
| `--img` | `1024` | Higher resolution detects small Class 0 hotspot defects |
| `--batch` | `4` | Fits T4 GPU at 1024 px with the `yolov5l` model |
| `--epochs` | `300` | Sufficient for small dataset with heavy augmentation |
| `--weights` | `yolov5l.pt` | Large model — better feature extraction, targets ≥ 75 % precision |
| `--optimizer` | `AdamW` | Stable convergence on small datasets |
| `--freeze` | `10` | Freeze first 10 backbone layers; preserve ImageNet features |
| `--hyp` | `hyp.scratch_custom.yaml` | Custom hyperparameters with label smoothing & heavy augmentation |
| `--cache` | `ram` | Cache images in RAM — eliminates disk I/O and cuts per-epoch time significantly |
| `--workers` | `8` | 8 parallel data-loader workers to keep the GPU fed |


In [ ]:
%cd /content/yolov5

!python train.py \
    --img 1024 \
    --batch 4 \
    --epochs 300 \
    --data /content/Activity_yolov5/personalizado.yaml \
    --weights yolov5l.pt \
    --optimizer AdamW \
    --freeze 10 \
    --hyp data/hyps/hyp.scratch_custom.yaml \
    --project /content/runs/train \
    --name solar_panels \
    --cache ram \
    --workers 8 \
    --exist-ok

## Step 8 — Review training results

In [ ]:
from IPython.display import Image as IPImage, display
import glob

results_dir = '/content/runs/train/solar_panels'

# Display training curves
for img_path in sorted(glob.glob(f'{results_dir}/*.png')):
    print(img_path)
    display(IPImage(img_path))

## Step 9 — Validate best model

In [ ]:
%cd /content/yolov5

!python val.py \
    --img 1024 \
    --data /content/Activity_yolov5/personalizado.yaml \
    --weights /content/runs/train/solar_panels/weights/best.pt \
    --task val \
    --verbose

## Step 9b — Precision gate (≥ 75 %)

Parse the validation results and assert that the model meets the required 75 % precision threshold.

In [ ]:
import csv
from pathlib import Path

PRECISION_THRESHOLD = 0.75
results_csv = Path('/content/runs/train/solar_panels/results.csv')

if not results_csv.exists():
    raise FileNotFoundError(f'Results file not found: {results_csv}')

# Read precision (column 'metrics/precision') from the last epoch row
with open(results_csv, newline='') as f:
    reader = csv.DictReader(f)
    rows = [row for row in reader]

if not rows:
    raise ValueError('results.csv is empty — training may not have completed.')

# Strip whitespace from column names and values
last_row = {k.strip(): v.strip() for k, v in rows[-1].items()}
precision_col = 'metrics/precision'

if precision_col not in last_row:
    raise KeyError(
        f"Column '{precision_col}' not found. Available columns: {list(last_row.keys())}"
    )

best_precision = float(last_row[precision_col])
print(f'Final epoch precision: {best_precision:.4f} ({best_precision * 100:.1f}%)')

if best_precision >= PRECISION_THRESHOLD:
    print(f'✅ PASS — Precision {best_precision * 100:.1f}% ≥ {PRECISION_THRESHOLD * 100:.0f}% threshold')
else:
    raise ValueError(
        f'❌ FAIL — Precision {best_precision * 100:.1f}% < {PRECISION_THRESHOLD * 100:.0f}% threshold.\n'
        'Consider: more training epochs, additional labelled data, or further hyperparameter tuning.'
    )


## Step 10 — Download trained weights

In [ ]:
from google.colab import files

files.download('/content/runs/train/solar_panels/weights/best.pt')